# 카페 프랜차이즈 인허가 데이터 → 대시보드 데이터 파이프라인

행정안전부 지방행정 인허가 데이터(식품업종)에서 브랜드명으로 매칭한 사업장 목록 CSV
10개(스타벅스, 이디야커피, 메가커피, 컴포즈커피, 투썸플레이스, 빽다방, 더벤티, 공차,
텐퍼센트커피, 매머드익스프레스)를 정제해, `coffee_dashboard.html`이 그대로 읽어들이는
`dashboard_data.json`을 만드는 전체 과정입니다.

**전체 흐름**
1. CSV 로드 & 정제 (매장 / 자판기·무인 / 제외 분류, 시도·시군구 파싱)
2. 브랜드별 요약 통계 (현재 매장 수, 폐업률, 생존기간)
3. 연도별 개업·폐업, 누적 매장 수 추이
4. 지역 분포 (시도별 집계, 시군구 TOP)
5. 생존기간 분포, 요일·월별 개업 패턴
6. 지도용 좌표 포인트
7. 스타벅스 근접 효과 (거리 기반 분석)
8. 최종 JSON 저장

각 셀은 순서대로 실행하면 됩니다. **셀 2의 `DATA_DIR`**만 CSV 파일이 있는 폴더로 바꿔주세요.


In [ ]:
import pandas as pd
import numpy as np
import json, re, math
from pathlib import Path

pd.set_option('display.max_columns', 100)


## 1. 설정

- `DATA_DIR`: 10개 브랜드 CSV(`{브랜드}_사업장목록.csv`)가 들어있는 폴더
- `OUTPUT_JSON`: 대시보드 HTML이 읽어들일 결과 파일 경로
- `BRAND_COLORS`: 각 브랜드의 실제 브랜드 컬러 (차트 색상으로 그대로 사용)
- `NOISE_KW`: 물류센터·공장·본사 등 실제 매장이 아닌 사업장명을 걸러내는 키워드
- `VENDING_KW`: 자판기/무인카페로 분류할 사업장명 키워드


In [ ]:
DATA_DIR = Path('./data')          # CSV 파일들이 있는 폴더로 변경하세요
OUTPUT_JSON = Path('./dashboard_data.json')

FILES = {
    '스타벅스': '스타벅스_사업장목록.csv',
    '이디야커피': '이디야커피_사업장목록.csv',
    '메가커피': '메가커피_사업장목록.csv',
    '컴포즈커피': '컴포즈커피_사업장목록.csv',
    '투썸플레이스': '투썸플레이스_사업장목록.csv',
    '빽다방': '빽다방_사업장목록.csv',
    '더벤티': '더벤티_사업장목록.csv',
    '공차': '공차_사업장목록.csv',
    '텐퍼센트커피': '텐퍼센트커피_사업장목록.csv',
    '매머드익스프레스': '매머드익스프레스_사업장목록.csv',
}

BRAND_COLORS = {
    '스타벅스': '#00704A', '이디야커피': '#1B4CA5', '메가커피': '#8DC63F',
    '컴포즈커피': '#5B3A29', '투썸플레이스': '#B8860B', '빽다방': '#E4572E',
    '더벤티': '#2AA9A0', '공차': '#8B1E3F', '텐퍼센트커피': '#C9A227',
    '매머드익스프레스': '#4B4E6D',
}

# 물류센터/공장/휴게소 등 명백히 '매장'이 아닌 사업장명 패턴 (인허가 데이터 특유의 노이즈)
NOISE_KW = re.compile('통운|물류|로지스틱|유통센터|도매|공장|R&D|오피스|본사|고속도로|휴게소')
# 자판기/무인카페로 분류할 사업장명 패턴
VENDING_KW = re.compile('캡슐|자판기|무인')


## 2. 브랜드별 CSV 로드 & 정제

각 브랜드 CSV는 행정안전부 인허가 데이터를 브랜드명으로 매칭한 것이라, 식품업종 전체가
섞여 있습니다(휴게음식점뿐 아니라 유통전문판매업, 축산물보관업 등도 포함). 아래 기준으로
세 가지로 분류합니다.

| 카테고리 | 기준 |
|---|---|
| **매장** | 업태(`_매칭업종`)가 휴게음식점·일반음식점·제과점영업 이면서, 사업장명에 물류/공장 키워드가 없는 경우 |
| **자판기/무인** | 업태가 식품자동판매기업이거나, 사업장명에 '캡슐·자판기·무인'이 포함된 경우 |
| **제외** | 그 외 (유통, 축산물보관, 즉석판매제조가공 등 — 매장이 아닌 인허가 건) |

시도명은 도로명주소(없으면 지번주소)의 첫 단어를 사용합니다. 단, 원본 데이터에 간혹
`전남광주통합특별시`처럼 전라남도/광주광역시가 뒤섞인 표기가 있어, 시군구가 광주의 구
(동구·서구·남구·북구·광산구)인지 여부로 다시 나눠줍니다.


In [ ]:
rows_ep = []   # 인허가 건(episode) 단위 데이터 — 폐업/재개업 이력까지 모두 포함

for brand, fname in FILES.items():
    df = pd.read_csv(DATA_DIR / fname, encoding='utf-8', low_memory=False)
    df['브랜드'] = brand

    biz = df['_매칭업종'].fillna('')
    name = df['사업장명'].fillna('').astype(str)

    is_store_biz = biz.isin(['휴게음식점', '일반음식점', '제과점영업'])
    is_vending = (biz == '식품자동판매기업') | name.str.contains(VENDING_KW)
    is_noise_name = name.str.contains(NOISE_KW)

    category = np.select(
        [is_vending, is_store_biz & ~is_noise_name, is_store_biz & is_noise_name],
        ['자판기/무인', '매장', '제외'],
        default='제외'
    )
    df['_카테고리'] = category
    df = df[df['_카테고리'] != '제외'].copy()

    df['_인허가일자_dt'] = pd.to_datetime(df['인허가일자'], errors='coerce')
    df['_폐업일자_dt'] = pd.to_datetime(df['폐업일자'], errors='coerce')
    df['_active'] = df['영업상태명'] == '영업/정상'
    df['_closed'] = df['영업상태명'] == '폐업'

    addr = df['도로명주소'].fillna(df['지번주소'])
    parts = addr.astype(str).str.split()
    df['_addr_norm'] = addr
    df['_sido'] = parts.str[0]
    df['_sigungu'] = parts.str[1]

    # '전남광주통합특별시' 표기 오류 보정
    gu_mask = df['_sido'] == '전남광주통합특별시'
    is_gwangju_gu = df['_sigungu'].isin(['동구', '서구', '남구', '북구', '광산구'])
    df.loc[gu_mask & is_gwangju_gu, '_sido'] = '광주광역시'
    df.loc[gu_mask & ~is_gwangju_gu, '_sido'] = '전라남도'

    keep_cols = ['브랜드', '사업장명', '_카테고리', '영업상태명', '_active', '_closed',
                 '_인허가일자_dt', '_폐업일자_dt', '_생존일수', '_경도', '_위도',
                 '_addr_norm', '_sido', '_sigungu']
    rows_ep.append(df[keep_cols])

ep = pd.concat(rows_ep, ignore_index=True)
ep['_year'] = ep['_인허가일자_dt'].dt.year
ep['_close_year'] = ep['_폐업일자_dt'].dt.year
ep['_month'] = ep['_인허가일자_dt'].dt.month
ep['_weekday'] = ep['_인허가일자_dt'].dt.day_name()

print('정제 후 총 인허가 건수:', len(ep))
ep.groupby(['브랜드', '_카테고리']).size()


## 3. 현재 매장(중복 제거) 만들기

같은 브랜드가 같은 주소에서 여러 번 인허가를 받는 경우가 있습니다(폐업 후 동일 자리 재개업,
업종 이중 등록 등). **매장 수 집계·지도**에는 브랜드+주소 기준으로 가장 최근 인허가 건 하나만
남기고, **개업/폐업/생존기간 분석**에는 인허가 이력 전체(episode)를 그대로 사용합니다.


In [ ]:
store_ep = ep[ep['_카테고리'] == '매장'].copy()
store_ep_sorted = store_ep.sort_values('_인허가일자_dt')
current = store_ep_sorted.drop_duplicates(subset=['브랜드', '_addr_norm'], keep='last')
current_active = current[current['_active']]

print('브랜드+주소 기준 현재 매장(중복제거):', len(current))
print('그 중 현재 영업중:', len(current_active))


## 4. 브랜드별 요약 통계 (현재 매장 수, 폐업률, 생존기간)

In [ ]:
summary = []
for brand in FILES:
    b_ep = store_ep[store_ep['브랜드'] == brand]
    b_vend = ep[(ep['브랜드'] == brand) & (ep['_카테고리'] == '자판기/무인')]
    b_cur = current[current['브랜드'] == brand]
    b_cur_active = b_cur[b_cur['_active']]
    closed = b_ep[b_ep['_closed']]
    surv = closed['_생존일수'].dropna()
    surv = surv[surv > 0]
    summary.append({
        'brand': brand,
        'total_episodes': int(len(b_ep)),
        'active_now': int(b_ep['_active'].sum()),
        'closed_total': int(b_ep['_closed'].sum()),
        'vending_count': int(len(b_vend)),
        'unique_locations_active': int(len(b_cur_active)),
        'closure_rate_pct': round(100 * b_ep['_closed'].sum() / max(len(b_ep), 1), 1),
        'median_survival_days': float(surv.median()) if len(surv) else None,
        'avg_survival_days': float(surv.mean()) if len(surv) else None,
    })

pd.DataFrame(summary).sort_values('active_now', ascending=False)


## 5. 연도별 개업·폐업, 누적 매장 수

1994~2026년 범위로 매년 개업/폐업 건수를 집계하고, `개업 - 폐업`을 누적해 브랜드별
순매장 수 성장 곡선을 만듭니다.


In [ ]:
years = list(range(1994, 2027))

yearly = []
for brand in FILES:
    b = store_ep[store_ep['브랜드'] == brand]
    opened_by_year = b['_year'].value_counts()
    closed_by_year = b['_close_year'].value_counts()
    for y in years:
        yearly.append({
            'brand': brand, 'year': y,
            'opened': int(opened_by_year.get(y, 0)),
            'closed': int(closed_by_year.get(y, 0)),
        })

cumulative = []
for brand in FILES:
    b = store_ep[store_ep['브랜드'] == brand]
    opened_by_year = b['_year'].value_counts()
    closed_by_year = b['_close_year'].value_counts()
    net = 0
    for y in years:
        net += int(opened_by_year.get(y, 0)) - int(closed_by_year.get(y, 0))
        cumulative.append({'brand': brand, 'year': y, 'cumulative_active': max(net, 0)})

print(len(yearly), '개 연도x브랜드 행 생성')


## 6. 지역 분포 — 시도별 집계 & 시군구 TOP

In [ ]:
regional = (current_active.groupby(['브랜드', '_sido']).size()
            .reset_index(name='count').rename(columns={'브랜드': 'brand', '_sido': 'sido'}))
regional = regional[regional['sido'].notna()]
regional_records = regional.to_dict(orient='records')

sigungu_all = (current_active.groupby(['_sido', '_sigungu']).size().reset_index(name='count')
               .sort_values('count', ascending=False))
sigungu_all['area'] = sigungu_all['_sido'].fillna('') + ' ' + sigungu_all['_sigungu'].fillna('')
top_sigungu_overall = sigungu_all.head(20)[['area', 'count']].to_dict(orient='records')

sigungu_by_brand = (current_active.groupby(['브랜드', '_sido', '_sigungu']).size()
                     .reset_index(name='count'))
sigungu_by_brand['area'] = sigungu_by_brand['_sido'].fillna('') + ' ' + sigungu_by_brand['_sigungu'].fillna('')
top_sigungu_by_brand = {}
for brand in FILES:
    sub = sigungu_by_brand[sigungu_by_brand['브랜드'] == brand].sort_values('count', ascending=False).head(10)
    top_sigungu_by_brand[brand] = sub[['area', 'count']].to_dict(orient='records')

top_sigungu_overall[:5]


## 7. 생존기간 분포, 요일·월별 개업 패턴

폐업한 매장만 골라, 개업부터 폐업까지 걸린 기간(`_생존일수`)을 6개 구간으로 나눕니다.
요일·월 분포는 인허가일자(`_인허가일자_dt`) 기준입니다.


In [ ]:
bins = [0, 90, 180, 365, 730, 1825, 10**6]
bin_labels = ['3개월 미만', '3~6개월', '6개월~1년', '1~2년', '2~5년', '5년 이상']
survival_hist = []
for brand in FILES:
    closed = store_ep[(store_ep['브랜드'] == brand) & (store_ep['_closed'])]
    surv = closed['_생존일수'].dropna()
    surv = surv[surv > 0]
    cats = pd.cut(surv, bins=bins, labels=bin_labels, right=False)
    counts = cats.value_counts().reindex(bin_labels, fill_value=0)
    for lbl, c in counts.items():
        survival_hist.append({'brand': brand, 'bucket': lbl, 'count': int(c)})

wd_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
wd_kr = {'Monday': '월', 'Tuesday': '화', 'Wednesday': '수', 'Thursday': '목',
         'Friday': '금', 'Saturday': '토', 'Sunday': '일'}
weekday = []
for brand in FILES:
    b = store_ep[store_ep['브랜드'] == brand]
    vc = b['_weekday'].value_counts()
    total = vc.sum()
    for wd in wd_order:
        weekday.append({'brand': brand, 'weekday': wd_kr[wd],
                         'pct': round(100 * vc.get(wd, 0) / total, 2) if total else 0})

monthly = []
for brand in FILES:
    b = store_ep[store_ep['브랜드'] == brand]
    vc = b['_month'].value_counts()
    total = vc.sum()
    for m in range(1, 13):
        monthly.append({'brand': brand, 'month': m,
                         'pct': round(100 * vc.get(m, 0) / total, 2) if total else 0})


## 8. 지도용 좌표 포인트

대시보드 지도(산점도 + 시도별 단계구분도)에 쓸 개별 매장 좌표입니다. 용량을 줄이기 위해
좌표는 소수 4자리(약 10m 오차)로 반올림하고, **현재 영업중인 매장만** 포함합니다.


In [ ]:
pts = current_active.dropna(subset=['_경도', '_위도'])[['브랜드', '_경도', '_위도']]
points = [{'b': r['브랜드'], 'x': round(r['_경도'], 4), 'y': round(r['_위도'], 4)}
          for _, r in pts.iterrows()]

vend_pts = ep[(ep['_카테고리'] == '자판기/무인') & ep['_active']].dropna(subset=['_경도', '_위도'])
vending_points = [{'b': r['브랜드'], 'x': round(r['_경도'], 4), 'y': round(r['_위도'], 4)}
                   for _, r in vend_pts.iterrows()]

print(len(points), '개 매장 좌표,', len(vending_points), '개 자판기/무인 좌표')


## 9. 스타벅스 근접 효과

"제3의 장소"로서 스타벅스가 유동인구를 만든다면, 그 주변에 다른 카페 브랜드가 따라붙는지를
확인합니다. 위경도를 평면 좌표(미터 단위)로 근사 변환한 뒤, `scipy.spatial.cKDTree`로 각
매장에서 가장 가까운 스타벅스까지의 거리를 구하고 구간별 비율을 계산합니다.

`pip install scipy` 가 필요합니다.


In [ ]:
from scipy.spatial import cKDTree

def to_xy(lon, lat, lat0=36.5):
    R = 6371000  # 지구 반지름(m)
    x = np.radians(lon) * R * math.cos(math.radians(lat0))
    y = np.radians(lat) * R
    return x, y

sb = current_active[current_active['브랜드'] == '스타벅스'].dropna(subset=['_경도', '_위도'])
sb_x, sb_y = to_xy(sb['_경도'].values, sb['_위도'].values)
sb_tree = cKDTree(np.c_[sb_x, sb_y])

proximity = []
dist_buckets = [(0, 250), (250, 500), (500, 1000), (1000, 2000), (2000, 10**8)]
bucket_labels = ['250m 이내', '250~500m', '500m~1km', '1~2km', '2km 이상']
for brand in FILES:
    if brand == '스타벅스':
        continue
    b = current_active[current_active['브랜드'] == brand].dropna(subset=['_경도', '_위도'])
    if len(b) == 0:
        continue
    bx, by = to_xy(b['_경도'].values, b['_위도'].values)
    dist, _ = sb_tree.query(np.c_[bx, by], k=1)
    total = len(dist)
    for (lo, hi), lbl in zip(dist_buckets, bucket_labels):
        cnt = int(((dist >= lo) & (dist < hi)).sum())
        proximity.append({'brand': brand, 'bucket': lbl, 'pct': round(100 * cnt / total, 1)})

pd.DataFrame(proximity).pivot(index='brand', columns='bucket', values='pct')


## 10. 최종 JSON 저장

`coffee_dashboard.html`이 그대로 읽어들이는 형식으로 저장합니다.


In [ ]:
output = {
    'brands': list(FILES.keys()),
    'colors': BRAND_COLORS,
    'summary': summary,
    'yearly': yearly,
    'cumulative': cumulative,
    'regional': regional_records,
    'top_sigungu_overall': top_sigungu_overall,
    'top_sigungu_by_brand': top_sigungu_by_brand,
    'survival_hist': survival_hist,
    'weekday': weekday,
    'monthly': monthly,
    'points': points,
    'vending_points': vending_points,
    'proximity': proximity,
    'generated_note': '행정안전부 지방행정 인허가 데이터(식품업종) 기반, 브랜드명 매칭 및 정제 처리됨',
}

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False)

print('저장 완료:', OUTPUT_JSON.resolve())


## (선택) 한국 시도 경계 GeoJSON 준비

대시보드의 지도 시각화(ECharts `registerMap`)는 시/도 경계 GeoJSON이 별도로 필요합니다.
아래는 오픈소스 `southkorea/southkorea-maps` 저장소의 시도 경계를 내려받아 용량을 줄이고,
2023~2024년 개편된 시도명(강원특별자치도·전북특별자치도)으로 이름을 맞추는 과정입니다.
`Node.js`와 인터넷 연결이 필요하며, 대시보드 HTML을 새로 만들 때 한 번만 실행하면 됩니다.


In [ ]:
# !npm install -g mapshaper
# !curl -sL -o skorea-provinces.json "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json"
# !npx mapshaper skorea-provinces.json -simplify 8% -o format=geojson precision=0.001 skorea-simplified.json

# import json
# geo = json.load(open('skorea-simplified.json', encoding='utf-8'))
# rename = {'강원도': '강원특별자치도', '전라북도': '전북특별자치도'}
# for feat in geo['features']:
#     n = feat['properties']['name']
#     if n in rename:
#         feat['properties']['name'] = rename[n]
# json.dump(geo, open('skorea-simplified.json', 'w', encoding='utf-8'), ensure_ascii=False)


## (선택) 대시보드 HTML에 데이터 주입

`dashboard_data.json`과 `skorea-simplified.json`을 템플릿의 자리표시자에 넣어 완성된
단일 HTML 파일을 만드는 단계입니다. `template.html`은 `/*__DATA_JSON__*/`,
`/*__GEO_JSON__*/` 두 자리표시자를 포함하고 있어야 합니다.


In [ ]:
# with open('template.html', encoding='utf-8') as f:
#     html = f.read()
# with open('dashboard_data.json', encoding='utf-8') as f:
#     data_json = f.read()
# with open('skorea-simplified.json', encoding='utf-8') as f:
#     geo_json = f.read()
#
# html = html.replace('/*__DATA_JSON__*/', data_json).replace('/*__GEO_JSON__*/', geo_json)
#
# with open('coffee_dashboard.html', 'w', encoding='utf-8') as f:
#     f.write(html)
